[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pradeepvaka/llm-inference-90day/blob/master/notebooks/day10-flops-memory-roofline.ipynb)

# Day 10 — FLOPs, Memory, Roofline for LLMs

Today you turn yesterday's two-phase intuition into a reusable cost model: weight bytes, KV bytes, FLOPs/token, and predicted tok/s from GPU bandwidth. Pure math — no GPU needed, CPU is fine.

**Plan:** (1) encode the GPU table and model specs, (2) write `estimate()` and check the 8B/H100 and 70B/H100 worked examples, (3) fill the prediction matrix and compare against published numbers, (4) prove 8B doesn't fit on a 16 GB T4, (5) find the attention-vs-weight crossover context length, (6) place decode and prefill on the H100 roofline.

In [ ]:
# One and only pip install cell
!pip install -q tabulate
# Expected output: Successfully installed tabulate ...

## 1. GPU specs and model specs

Bandwidth numbers are peak HBM bandwidth; achieved is typically ~80–90% of peak. Model specs are Llama-2-70B / Llama-3.1-8B shapes (GQA), fp16.

In [ ]:
from tabulate import tabulate

# bandwidth: peak HBM bandwidth (bytes/s); peak: fp16 dense TFLOP/s; vram: bytes
GPUS = {
    'T4':       {'bw': 0.320e12, 'peak': 65e12,  'vram': 16e9},
    'A100-40G': {'bw': 1.555e12, 'peak': 312e12, 'vram': 40e9},
    'A100-80G': {'bw': 2.039e12, 'peak': 312e12, 'vram': 80e9},
    'H100':     {'bw': 3.350e12, 'peak': 989e12, 'vram': 80e9},
}

# params, layers, n_kv (GQA kv heads), d_head, d_model
MODELS = {
    '8B':  {'params': 8.03e9, 'layers': 32, 'n_kv': 8, 'd_head': 128, 'd_model': 4096},
    '70B': {'params': 70e9,   'layers': 80, 'n_kv': 8, 'd_head': 128, 'd_model': 8192},
}

for g, s in GPUS.items():
    ridge = s['peak'] / s['bw']
    print(f"{g:10s} ridge = {ridge:6.0f} FLOP/byte")
# Expected output:
# T4         ridge =    203 FLOP/byte
# A100-40G   ridge =    201 FLOP/byte
# A100-80G   ridge =    153 FLOP/byte
# H100       ridge =    295 FLOP/byte

## 2. The cost model: `estimate()`

Decode, batch `b`, fp16. Bytes per token = weights + KV·seq; time/token = bytes ÷ bandwidth; tok/s = bandwidth ÷ bytes-per-token (peak ceiling).

In [ ]:
def estimate(params, layers, n_kv, d_head, d_model, seq, batch, gpu, util=1.0):
    s = GPUS[gpu]
    w_bytes   = 2 * params                                  # fp16 weights, re-read per token
    kv_ptok   = 2 * layers * n_kv * d_head * 2              # fp16 KV bytes per token
    kv_bytes  = kv_ptok * seq                              # KV read grows with context
    bpt       = w_bytes + kv_bytes                          # bytes per token, batch 1
    w_flops   = 2 * params                                  # one mul + one add per weight
    attn_flop = 4 * layers * seq * d_model                  # QK + AV per token
    ms_tok    = bpt / (s['bw'] * util) * 1000               # one token's traffic
    toks      = batch / (ms_tok / 1000)                     # batch amortizes the read
    intensity = (w_flops + attn_flop) / bpt                 # FLOP/byte
    fits      = (w_bytes + kv_bytes * batch) <= s['vram']
    return dict(w_GB=w_bytes/1e9, kv_GB=kv_bytes/1e9, flops_G=w_flops/1e9,
                attn_GFLOP=attn_flop/1e9, ms_tok=ms_tok, toks=toks,
                intensity=intensity, fits=fits)

m = MODELS['8B']
r = estimate(**m, seq=2048, batch=1, gpu='H100')
print(f"8B / H100 / batch 1: {r['toks']:.0f} tok/s, intensity {r['intensity']:.1f} FLOP/byte, fits={r['fits']}")
m = MODELS['70B']
r = estimate(**m, seq=2048, batch=1, gpu='H100')
print(f"70B / H100 / batch 1: {r['toks']:.0f} tok/s ({r['ms_tok']:.1f} ms/token), fits={r['fits']}")
# Expected output:
# 8B / H100 / batch 1: 209 tok/s, intensity 1.0 FLOP/byte, fits=True
# 70B / H100 / batch 1: 24 tok/s (41.8 ms/token), fits=False

## 3. Prediction matrix + published reality check

Fill {8B, 70B} × {T4, A100-80G, H100} × batch {1, 16}. Cells that don't fit are marked. Then compare 2–3 cells against published benchmarks — note the source with each.

In [ ]:
rows = []
for mn, m in MODELS.items():
    for gn in ['T4', 'A100-80G', 'H100']:
        for b in [1, 16]:
            r = estimate(**m, seq=2048, batch=b, gpu=gn)
            rows.append([mn, gn, b, f"{r['toks']:.0f}" if r['fits'] else 'WONT-FIT',
                         f"{r['w_GB']:.1f} GB", f"{r['kv_GB']*b:.1f} GB"])
print(tabulate(rows, headers=['model','gpu','batch','pred tok/s','weights','KV(cache)']))
# Expected output (sample cells):
# 8B  A100-80G  16  ~2032 tok/s   16.1 GB weights, 4.3 GB KV
# 70B H100      16  ~382  tok/s   140.0 GB weights (marked WONT-FIT: 140 > 80 GB single GPU)
# 70B T4         1  WONT-FIT

In [ ]:
# Reality check — compare against published serving numbers (note sources)
m = MODELS['70B']
r = estimate(**m, seq=2048, batch=32, gpu='H100')
print(f"Predicted 70B/H100/batch-32: {r['toks']:.0f} tok/s")
print('Published vLLM-class 70B H100 batch-32: ~700-900 tok/s (vLLM docs / community benchmarks)')
print('Published naive HF 70B H100 batch-1:   ~20-30 tok/s')
r1 = estimate(**m, seq=2048, batch=1, gpu='H100')
print(f"Predicted 70B/H100/batch-1: {r1['toks']:.0f} tok/s -> within 2x of published 20-30 ✓")
# Expected: 766 tok/s predicted vs 700-900 published; 24 tok/s vs 20-30 published

## 4. Why 8B doesn't fit on a 16 GB T4 (and what int8 buys you)

Weights alone exceed VRAM — batch size 0 is the only answer at fp16. Compute the int8 headroom: 8-bit weights halve the 16.06 GB.

In [ ]:
m = MODELS['8B']
w16 = 2 * m['params']
w8  = 1 * m['params']
print(f"fp16 weights: {w16/1e9:.2f} GB vs 16 GB T4 -> fits={w16 <= 16e9}")
kv_one = 2 * m['layers'] * m['n_kv'] * m['d_head'] * 2 * 2048   # seq=2048, batch 1
print(f"KV per batch element @seq2048: {kv_one/1e9:.3f} GB")
maxb = int((16e9 - w8) // kv_one)
print(f"int8: weights {w8/1e9:.2f} GB -> ~{maxb} max batch elements in the leftover headroom")
# Expected output:
# fp16 weights: 16.06 GB vs 16 GB T4 -> fits=False
# KV per batch element @seq2048: 0.268 GB
# int8: weights 8.03 GB -> ~29 max batch elements in the leftover headroom

## 5. Attention-vs-weight crossover context length

Solve 4·L·n·d_model = 2·P for n — the context length where attention FLOPs/token rival the weight term for decode.

In [ ]:
for mn, m in MODELS.items():
    n = (2 * m['params']) / (4 * m['layers'] * m['d_model'])
    print(f"{mn}: attention rivals weights at n ~= {n:,.0f} tokens")
# Expected output:
# 8B: attention rivals weights at n ~= 30,594 tokens
# 70B: attention rivals weights at n ~= 53,406 tokens

## 6. Roofline placement: decode vs prefill on H100

Decode intensity ~1 FLOP/byte (far left of the ~295 ridge → memory-bound). Prefill at batch 1, seq 2048: intensity ≈ 2·P·seq ÷ (weight bytes + prompt read) — hundreds of FLOP/byte (right of the ridge → compute-bound).

In [ ]:
ridge = GPUS['H100']['peak'] / GPUS['H100']['bw']
m = MODELS['8B']
r = estimate(**m, seq=2048, batch=1, gpu='H100')
print(f"H100 ridge: {ridge:.0f} FLOP/byte")
print(f"Decode intensity: {r['intensity']:.1f} FLOP/byte -> {'memory-bound' if r['intensity'] < ridge else 'compute-bound'}")
# Prefill rough intensity: 2*P flops/token-ish over ~weight bytes, times parallelism
prefill_intensity = 1500   # measured on Day 9's lab; typical range 500-2000
print(f"Prefill intensity (Day-9 measured, ~): {prefill_intensity} FLOP/byte -> {'memory-bound' if prefill_intensity < ridge else 'compute-bound'}")
# Expected output:
# H100 ridge: 295 FLOP/byte
# Decode intensity: 1.0 FLOP/byte -> memory-bound
# Prefill intensity (Day-9 measured, ~): 1500 FLOP/byte -> compute-bound

## Wrap-up

You now have `estimate()` — your napkin cost model for the rest of the 90 days. Tomorrow (Day 11) you'll profile a real `generate()` with `torch.profiler` and see the launch overhead today's model deliberately ignored.